# 05 — Evaluasi Model
**Dataset:** Fruits 360  
**Tujuan Notebook:** Evaluasi lengkap model terbaik menggunakan berbagai metrik dan visualisasi.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pickle

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix
)
import warnings
warnings.filterwarnings('ignore')

OUTPUT_DIR = 'data/processed'
MODEL_DIR  = 'models'
print('Libraries loaded!')

## 2. Load Model & Data

In [ ]:
with open(f'{MODEL_DIR}/best_model.pkl', 'rb') as f:
    model = pickle.load(f)
with open(f'{MODEL_DIR}/scaler.pkl', 'rb') as f:
    scaler = pickle.load(f)

X_train = np.load(f'{OUTPUT_DIR}/X_train_pca.npy')
X_test  = np.load(f'{OUTPUT_DIR}/X_test_pca.npy')
y_train = np.load(f'{OUTPUT_DIR}/y_train.npy')
y_test  = np.load(f'{OUTPUT_DIR}/y_test.npy')
label_names = pd.read_csv(f'{OUTPUT_DIR}/label_names.csv')['class_name'].tolist()

X_train_scaled = scaler.transform(X_train)
X_test_scaled  = scaler.transform(X_test)

y_pred = model.predict(X_test_scaled)
print('Model dan data berhasil di-load!')

## 3. Metrik Evaluasi Utama

In [ ]:
acc  = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred, average='weighted', zero_division=0)
rec  = recall_score(y_test, y_pred, average='weighted', zero_division=0)
f1   = f1_score(y_test, y_pred, average='weighted', zero_division=0)

metrics_df = pd.DataFrame({
    'Metrik': ['Accuracy', 'Precision (weighted)', 'Recall (weighted)', 'F1-Score (weighted)'],
    'Nilai':  [acc, prec, rec, f1]
})

print('=== Hasil Evaluasi Model Terbaik ===')
print(metrics_df.to_string(index=False))

# Bar chart metrik
plt.figure(figsize=(8, 4))
colors = ['steelblue', 'coral', 'mediumseagreen', 'mediumpurple']
plt.bar(metrics_df['Metrik'], metrics_df['Nilai'], color=colors)
plt.ylim(0, 1.1)
plt.title('Metrik Evaluasi Model')
plt.ylabel('Score')
for i, v in enumerate(metrics_df['Nilai']):
    plt.text(i, v + 0.02, f'{v:.4f}', ha='center', fontsize=10)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/evaluation_metrics.png', dpi=100)
plt.show()

## 4. Classification Report

In [ ]:
# Tampilkan laporan lengkap per kelas
unique_labels = np.unique(y_test)
target_names  = [label_names[i] for i in unique_labels]

report = classification_report(y_test, y_pred, labels=unique_labels, target_names=target_names, zero_division=0)
print('=== Classification Report ===')
print(report)

## 5. Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred, labels=unique_labels)

# Plot confusion matrix (hanya tampilkan sebagian jika kelas terlalu banyak)
n_show = min(30, len(unique_labels))
cm_show = cm[:n_show, :n_show]
names_show = target_names[:n_show]

plt.figure(figsize=(16, 14))
sns.heatmap(
    cm_show,
    annot=True, fmt='d', cmap='Blues',
    xticklabels=names_show,
    yticklabels=names_show,
    linewidths=0.5
)
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title(f'Confusion Matrix (30 Kelas Pertama)')
plt.xticks(rotation=45, ha='right', fontsize=7)
plt.yticks(fontsize=7)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/confusion_matrix.png', dpi=100)
plt.show()

## 6. Per-Class Accuracy (Top & Bottom)

In [ ]:
per_class_acc = cm.diagonal() / cm.sum(axis=1)
class_acc_df  = pd.DataFrame({'Class': target_names, 'Accuracy': per_class_acc})
class_acc_df  = class_acc_df.sort_values('Accuracy', ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Top 10 kelas
top10 = class_acc_df.head(10)
axes[0].barh(top10['Class'], top10['Accuracy'], color='mediumseagreen')
axes[0].set_title('Top 10 Kelas (Accuracy Tertinggi)')
axes[0].set_xlim(0, 1)

# Bottom 10 kelas
bot10 = class_acc_df.tail(10)
axes[1].barh(bot10['Class'], bot10['Accuracy'], color='coral')
axes[1].set_title('Bottom 10 Kelas (Accuracy Terendah)')
axes[1].set_xlim(0, 1)

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/per_class_accuracy.png', dpi=100)
plt.show()

## 7. Ringkasan
- Model berhasil dievaluasi dengan metrik Accuracy, Precision, Recall, dan F1-Score
- Confusion Matrix menunjukkan performa per kelas
- Kelas dengan accuracy rendah umumnya memiliki kemiripan visual antar buah
- Lanjut ke notebook `06_mlflow_tracking.ipynb` untuk logging eksperimen